In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

# 1. Setup & Reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Metadata & Grouped Split
meta_path = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)[0]
df = pd.read_csv(meta_path)
img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(os.path.dirname(meta_path), "**", "*.jpg"), recursive=True)}

df["path"] = df["image_id"].map(img_map)
df = df.dropna(subset=["path"]).reset_index(drop=True)

LABEL_MAP = {"nv": 0, "mel": 1, "bkl": 2, "bcc": 3, "akiec": 4, "vasc": 5, "df": 6}
CLASS_NAMES = list(LABEL_MAP.keys())
df["label"] = df["dx"].map(LABEL_MAP)

sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
splits = list(sgkf.split(df, df["label"], groups=df["lesion_id"]))

test_df = df.iloc[splits[0][1]].reset_index(drop=True)
val_df = df.iloc[splits[1][1]].reset_index(drop=True)
train_df = df.drop(index=list(splits[0][1]) + list(splits[1][1])).reset_index(drop=True)

# 3. Data Transforms & Loaders
IMG_SIZE = 256
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=180, border_mode=cv2.BORDER_REFLECT, p=0.6),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, p=0.4),
    A.Normalize(),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2()
])

class SkinDataset(Dataset):
    def __init__(self, data, transform=None):
        self.paths = data["path"].values
        self.labels = data["label"].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(SkinDataset(train_df, train_tf), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(SkinDataset(val_df, val_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(SkinDataset(test_df, val_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# 4. Model, Loss & Optimizer
model = timm.create_model("convnext_small.fb_in22k_ft_in1k", pretrained=True, num_classes=len(CLASS_NAMES), drop_rate=0.3).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

# 5. Training Loop
EPOCHS = 15
best_acc = 0.0
save_path = "/kaggle/working/best_model_full.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    scheduler.step()

    # Evaluation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_targets.extend(labels.numpy())

    acc = accuracy_score(val_targets, val_preds)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Val Accuracy: {acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), save_path)
        print(f"  Saved best model ({best_acc*100:.2f}%)")

# 6. Test Inference with Simple TTA (Flips)
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda'):
            # Average original, horizontal flip, and vertical flip
            p1 = torch.softmax(model(imgs), dim=1)
            p2 = torch.softmax(model(torch.flip(imgs, dims=[3])), dim=1)
            p3 = torch.softmax(model(torch.flip(imgs, dims=[2])), dim=1)
            probs = (p1 + p2 + p3) / 3.0

        y_pred.extend(probs.argmax(dim=1).cpu().numpy())
        y_true.extend(labels.numpy())

print("\n" + "=" * 40)
print(f"Final Test Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred)*100:.2f}%")
print("=" * 40)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

Epoch 01/15 | Val Accuracy: 79.48%
  Saved best model (79.48%)
Epoch 02/15 | Val Accuracy: 84.36%
  Saved best model (84.36%)
Epoch 03/15 | Val Accuracy: 83.76%
Epoch 04/15 | Val Accuracy: 84.76%
  Saved best model (84.76%)
Epoch 05/15 | Val Accuracy: 84.26%
Epoch 06/15 | Val Accuracy: 86.06%
  Saved best model (86.06%)
Epoch 07/15 | Val Accuracy: 87.65%
  Saved best model (87.65%)
Epoch 08/15 | Val Accuracy: 84.26%
Epoch 09/15 | Val Accuracy: 85.86%
Epoch 10/15 | Val Accuracy: 85.76%
Epoch 11/15 | Val Accuracy: 86.65%
Epoch 12/15 | Val Accuracy: 85.56%
Epoch 13/15 | Val Accuracy: 86.16%
Epoch 14/15 | Val Accuracy: 86.95%
Epoch 15/15 | Val Accuracy: 86.55%

========================================
Final Test Accuracy: 87.98%
Balanced Accuracy: 74.76%
========================================
              precision    recall  f1-score   support

          nv     0.9483    0.9649    0.9565       684
         mel     0.7160    0.5524    0.6237       105
         bkl     0.7518    0.7923    0.7715       130
         bcc     0.6786    0.9744    0.8000        39
       akiec     0.6667    0.5000    0.5714        28
        vasc     0.7333    0.6875    0.7097        16
          df     0.9412    0.7619    0.8421        21

    accuracy                         0.8798      1023
   macro avg     0.7766    0.7476    0.7536      1023
weighted avg     0.8780    0.8798    0.8761      1023
